# Out-of-Sample Predictability: 9 MOIs

Validate predictability of post-stimulus metrics from pre-stimulus metrics using within-session K-fold cross-validation.

## Problem Statement

- **Current analysis**: Within-sample correlation between xᵢ (pre-stimulus) and yᵢ (post-stimulus) is descriptive only.
- **Reviewer's concern**: Claiming "prediction" requires out-of-sample validation.
- **Solution**: Within-session K-fold CV with held-out trial evaluation.

## Method

For each session and metric:
1. Split N trials into K folds (e.g., 5-fold)
2. For each fold: train OLS on (K−1) folds, test on 1 held-out fold
3. Report out-of-sample R² and correlation (true predictability)
4. Compare to within-sample correlation (apparent vs. real)

In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr, pearsonr
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Metrics of interest (9 MOIs) in raw/source names
METRICS_9_ORDER = [
    'salience',
    'dynamic_functional_connectivity_matrix_var_mat',
    'max_dynamic_network_profile',
    'Entropy_FC',
    'phase_coherence_matrix_mean_mat',
    'N1',
    'N2',
]

# Display labels
METRICS_9_LABELS = [
    'Salience',
    'Fluidity',
    'Peak Synchrony',
    'Complexity',
    'Entrainment',
    'N1_amplitude',
    'N2_amplitude',
]

METRIC_LABEL_MAP = dict(zip(METRICS_9_ORDER, METRICS_9_LABELS))

In [4]:
# Resolve project root
cwd = Path.cwd().resolve()
if (cwd / 'data').exists():
    project_root = cwd
elif (cwd.parent / 'data').exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError('Could not find project root containing data/.')

metrics_root = project_root / 'data' / 'df_results' / 'ieeg_metrics'
print(f'Project root: {project_root}')
print(f'Metrics root: {metrics_root}')

Project root: /Users/cbc/Documents/GitHub/fufo/notebook/DavideMomi/Revision/State_Dependent_Brain_Stimulation-main
Metrics root: /Users/cbc/Documents/GitHub/fufo/notebook/DavideMomi/Revision/State_Dependent_Brain_Stimulation-main/data/df_results/ieeg_metrics


In [5]:
# Load df_9MOIs
csv_path_9mois = metrics_root / 'df_9MOIs_ieeg.csv'
df_9MOIs = pd.read_csv(csv_path_9mois)

# Convert metric_name to ordered categorical
df_9MOIs['metric_name'] = pd.Categorical(
    df_9MOIs['metric_name'],
    categories=METRICS_9_ORDER,
    ordered=True,
)

print(f'Loaded df_9MOIs: {df_9MOIs.shape}')
print(f'Columns: {list(df_9MOIs.columns)}')

Loaded df_9MOIs: (3246210, 8)
Columns: ['sub', 'radius_pre', 'radius_post', 'run_is', 'trial_id', 'metric_name', 'metric_value_pre', 'metric_value_post']


In [6]:
# Filter for radius_pre=100, radius_post=100 only
df_filtered = df_9MOIs[
    (df_9MOIs['radius_pre'] == 100) & (df_9MOIs['radius_post'] == 100)
].copy()

print(f'After filtering (radius_pre=100, radius_post=100): {df_filtered.shape}')
print(f'Unique subjects: {df_filtered["sub"].nunique()}')
print(f'Unique sessions: {df_filtered["run_is"].nunique()}')
print(f'Unique metrics: {df_filtered["metric_name"].nunique()}')

After filtering (radius_pre=100, radius_post=100): (98370, 8)
Unique subjects: 36
Unique sessions: 17
Unique metrics: 7


In [7]:
def compute_cv_score(x, y, n_splits=5, random_state=42, metric='r2'):
    """
    Compute out-of-sample R² or correlation using K-fold cross-validation.
    
    Parameters:
    -----------
    x : array-like, shape (n_samples,)
        Pre-stimulus metric values
    y : array-like, shape (n_samples,)
        Post-stimulus metric values
    n_splits : int
        Number of folds
    random_state : int
        Random seed for reproducibility
    metric : {'r2', 'corr'}
        Evaluation metric (out-of-sample R² or Pearson correlation)
    
    Returns:
    --------
    mean_score : float
        Mean out-of-sample score across folds
    std_score : float
        Std dev of out-of-sample scores
    within_sample_corr : float
        Within-sample Pearson correlation (for comparison)
    """
    x = np.asarray(x).flatten()
    y = np.asarray(y).flatten()
    
    # Remove NaN pairs
    mask = ~(np.isnan(x) | np.isnan(y))
    x = x[mask]
    y = y[mask]
    
    if len(x) < n_splits or len(x) < 3:
        return np.nan, np.nan, np.nan
    
    # Within-sample correlation (for comparison)
    within_sample_corr, _ = pearsonr(x, y)
    
    # K-fold cross-validation
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    scores = []
    
    for train_idx, test_idx in kf.split(x):
        X_train, X_test = x[train_idx].reshape(-1, 1), x[test_idx].reshape(-1, 1)
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Fit OLS model
        model = LinearRegression()
        model.fit(X_train, y_train)
        
        # Evaluate on held-out fold
        y_pred = model.predict(X_test)
        
        if metric == 'r2':
            # Out-of-sample R²
            ss_res = np.sum((y_test - y_pred) ** 2)
            ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
            r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else np.nan
            scores.append(r2)
        elif metric == 'corr':
            # Out-of-sample Pearson correlation
            corr, _ = pearsonr(y_test, y_pred)
            scores.append(corr)
    
    scores = np.array(scores)
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    
    return mean_score, std_score, within_sample_corr

In [8]:
# Build predictability dataframe
# For each (subject, session, metric) triple: compute CV scores

results = []

group_cols = ['sub', 'run_is', 'metric_name']
n_groups = df_filtered.groupby(group_cols).ngroups

print(f'Computing CV scores for {n_groups} (subject, session, metric) groups...\n')

for (sub, run, metric), group_df in df_filtered.groupby(group_cols, dropna=False):
    # Extract pre (x) and post (y) values for this group
    x = group_df['metric_value_pre'].values
    y = group_df['metric_value_post'].values
    trial_ids = group_df['trial_id'].values
    
    n_trials = len(x)
    
    # Compute out-of-sample R²
    oos_r2, oos_r2_std, within_corr = compute_cv_score(
        x, y, n_splits=min(5, max(2, n_trials // 3)), metric='r2'
    )
    
    # Compute out-of-sample correlation
    oos_corr, oos_corr_std, _ = compute_cv_score(
        x, y, n_splits=min(5, max(2, n_trials // 3)), metric='corr'
    )
    
    # Store result
    results.append({
        'sub': sub,
        'run_is': run,
        'metric_name': metric,
        'metric_label': METRIC_LABEL_MAP.get(metric, metric),
        'n_trials': n_trials,
        'within_sample_corr': within_corr,
        'oos_r2': oos_r2,
        'oos_r2_std': oos_r2_std,
        'oos_corr': oos_corr,
        'oos_corr_std': oos_corr_std,
    })

# Create results dataframe
df_predictability = pd.DataFrame(results)

print(f'Computed predictability scores for {len(df_predictability)} groups')
print(f'\nDataframe shape: {df_predictability.shape}')
print(f'\nFirst 10 rows:')
print(df_predictability.head(10))

Computing CV scores for 2226 (subject, session, metric) groups...

Computed predictability scores for 2544 groups

Dataframe shape: (2544, 10)

First 10 rows:
      sub  run_is                                     metric_name  \
0  sub-01  run-01                                        salience   
1  sub-01  run-01  dynamic_functional_connectivity_matrix_var_mat   
2  sub-01  run-01                     max_dynamic_network_profile   
3  sub-01  run-01                                      Entropy_FC   
4  sub-01  run-01                 phase_coherence_matrix_mean_mat   
5  sub-01  run-01                                              N1   
6  sub-01  run-01                                              N2   
7  sub-01  run-01                                             NaN   
8  sub-01  run-02                                        salience   
9  sub-01  run-02  dynamic_functional_connectivity_matrix_var_mat   

     metric_label  n_trials  within_sample_corr    oos_r2  oos_r2_std  \
0       

In [9]:
# Summary statistics: compare within-sample vs. out-of-sample
print('\n' + '='*70)
print('SUMMARY: Within-Sample vs. Out-of-Sample Predictability')
print('='*70)
print(f'\nMedian within-sample correlation: {df_predictability["within_sample_corr"].median():.3f}')
print(f'Median out-of-sample R²: {df_predictability["oos_r2"].median():.3f}')
print(f'Median out-of-sample correlation: {df_predictability["oos_corr"].median():.3f}')

print(f'\nWithin-sample range: [{df_predictability["within_sample_corr"].min():.3f}, {df_predictability["within_sample_corr"].max():.3f}]')
print(f'Out-of-sample R² range: [{df_predictability["oos_r2"].min():.3f}, {df_predictability["oos_r2"].max():.3f}]')
print(f'Out-of-sample corr range: [{df_predictability["oos_corr"].min():.3f}, {df_predictability["oos_corr"].max():.3f}]')

# How many sessions show degradation from within to out-of-sample?
degradation = (df_predictability['within_sample_corr'] - df_predictability['oos_corr']).mean()
print(f'\nMean degradation (within - out-of-sample): {degradation:.3f}')

# Proportion with positive out-of-sample performance
pct_positive_oos = (df_predictability['oos_r2'] > 0).sum() / len(df_predictability) * 100
print(f'Fraction with positive out-of-sample R²: {pct_positive_oos:.1f}%')


SUMMARY: Within-Sample vs. Out-of-Sample Predictability

Median within-sample correlation: 0.367
Median out-of-sample R²: -0.292
Median out-of-sample correlation: 0.366

Within-sample range: [-0.661, 0.983]
Out-of-sample R² range: [-4994.453, 0.958]
Out-of-sample corr range: [-0.895, 0.983]

Mean degradation (within - out-of-sample): 0.037
Fraction with positive out-of-sample R²: 24.9%


In [10]:
# Save results
output_csv = metrics_root / 'df_predictability_9MOIs.csv'
df_predictability.to_csv(output_csv, index=False)
print(f'Saved: {output_csv}')
print(f'Shape: {df_predictability.shape}')

Saved: /Users/cbc/Documents/GitHub/fufo/notebook/DavideMomi/Revision/State_Dependent_Brain_Stimulation-main/data/df_results/ieeg_metrics/df_predictability_9MOIs.csv
Shape: (2544, 10)


In [11]:
# Per-metric summary
print('\n' + '='*70)
print('Per-Metric Summary')
print('='*70)

metric_summary = df_predictability.groupby('metric_label').agg({
    'within_sample_corr': ['mean', 'median', 'std'],
    'oos_r2': ['mean', 'median', 'std'],
    'oos_corr': ['mean', 'median', 'std'],
    'n_trials': 'mean'
}).round(3)

print(metric_summary)


Per-Metric Summary
               within_sample_corr                oos_r2                  \
                             mean median    std    mean median      std   
metric_label                                                              
Complexity                  0.279  0.301  0.230  -0.415 -0.233    0.664   
Entrainment                 0.281  0.285  0.239  -0.389 -0.226    0.682   
Fluidity                    0.306  0.317  0.236  -0.421 -0.229    0.819   
N1_amplitude                0.208  0.197  0.236 -31.277 -1.754  288.494   
N2_amplitude                0.263  0.257  0.232  -8.386 -1.251   22.400   
Peak Synchrony              0.370  0.362  0.248  -2.460 -0.517    7.436   
Salience                    0.540  0.573  0.208  -0.390 -0.133    1.725   

               oos_corr               n_trials  
                   mean median    std     mean  
metric_label                                    
Complexity        0.231  0.274  0.278   34.371  
Entrainment       0.223  0.262  0